In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import glob
from xgrads import open_CtlDataset
from pathlib import Path
import netCDF4

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib.colors as mcolors

import ipywidgets as widgets
from IPython.display import display, clear_output

ncl_cmap = LinearSegmentedColormap.from_list(
    "BlueWhiteOrangeRed",
    ["#2166ac", "#67a9cf", "#ffffff", "#fdae61", "#b2182b"],
    N=256
)


plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_colwidth", 160)

print("Python OK")

Python OK


In [3]:
BASE_DIR = Path.cwd()

NEW_DIR = Path.cwd() / ".." / ".." / ".." / "SPEEDY_access" / "output" / "exp_102"
OLD_DIR = Path.cwd() / ".." / ".." / ".." / "SPEEDY_access" / "output" / "exp_101"
print("NEW_DIR:", NEW_DIR, NEW_DIR.exists())
print("OLD_DIR:", OLD_DIR, OLD_DIR.exists())

# print("\nNew files:")
# if NEW_DIR.exists():
#     for p in sorted(NEW_DIR.glob("*.grd")):
#         print(f"{p.name:25s} {p.stat().st_size / 1024**2:.2f} MB")
# else:
#     print("NEW_DIR does not exist")

# print("\nOld files:")
# if OLD_DIR.exists():
#     for p in sorted(OLD_DIR.glob("*.grd")):
#          print(f"{p.name:25s} {p.stat().st_size / 1024**2:.2f} MB")
# else:
#     print("OLD_DIR does not exist")   

SPEEDY_VARIABLE = "TEMP0"
ACCESS_VARIABLE = "tas"   

# Output directory
TAS_OUT_DIR = (Path.cwd() / ".." / ".." / "access_forcing").resolve()
TAS_OUT_DIR.mkdir(parents=True, exist_ok=True)
WRITE_ONE_FILE_PER_YEAR = True
# Keep the SPEEDY grid untouched until the JRA-55 metadata/grid are inspected.
SHIFT_LONGITUDE_TO_MINUS180_180 = False
SORT_LATITUDE_NORTH_TO_SOUTH = False
print("Output directory:", TAS_OUT_DIR)


NEW_DIR: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/scripts/access_forcing/../../../SPEEDY_access/output/exp_102 True
OLD_DIR: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/scripts/access_forcing/../../../SPEEDY_access/output/exp_101 True
Output directory: /leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing


In [4]:
for ctl in Path(NEW_DIR).glob("attm102.ctl"):
    print("=" * 80)
    print(ctl.name)

    ds = open_CtlDataset(str(ctl))
ds

attm102.ctl


<xarray.Dataset> Size: 17GB
Dimensions:  (time: 8760, lev: 8, lat: 48, lon: 96)
Coordinates:
  * time     (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lev      (lev) float64 64B 925.0 850.0 700.0 500.0 300.0 200.0 100.0 30.0
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Data variables: (12/43)
    GH       (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    TEMP     (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    U        (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    V        (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    Q        (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    RH       (time, lev, lat, lon) >f4 1GB dask.array<chunksize=(1, 1, 48, 96), meta=np.ndarray>
    ...       ...
    SHF      (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    LSHF     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SSHF     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SSRD     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SLRD     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    SNOW     (time, lat, lon) >f4 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
Attributes:
    comment:  geopotential height               [m]
    storage:  99
    title:    Means/variances
    undef:    9.999e+19
    pdef:     None

In [5]:
# =========================
# 2. Inspect source ST


if SPEEDY_VARIABLE not in ds:
    raise KeyError(
        f"{SPEEDY_VARIABLE!r} is absent from the SPEEDY dataset. "
        f"Available variables: {list(ds.data_vars)}"
    )

st = ds[SPEEDY_VARIABLE]

print(st)
print("\nDimensions:", st.dims)
print("Shape:", st.shape)
print("Dtype:", st.dtype)
print("Attributes:", st.attrs)

dt_hours = np.diff(ds.time.values) / np.timedelta64(1, "h")
print("\nUnique output intervals [hours]:", np.unique(dt_hours))

print(
    "\nTEMP0 range [K]:",
    float(st.min().compute()),
    "to",
    float(st.max().compute()),
)

print(
    "TEMP0  global mean [K]:",
    float(st.mean(skipna=True).compute()),
)

print("\nUnique output intervals [hours]:", np.unique(dt_hours))
print("First time:", ds.time.values[0])
print("Last time:", ds.time.values[-1])

<xarray.DataArray 'TEMP0' (time: 8760, lat: 48, lon: 96)> Size: 161MB
dask.array<reshape, shape=(8760, 48, 96), dtype=>f4, chunksize=(1, 48, 96), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lat      (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon      (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
Attributes:
    comment:  near-surface air temperature   [degK]
    storage:  99

Dimensions: ('time', 'lat', 'lon')
Shape: (8760, 48, 96)
Dtype: >f4
Attributes: {'comment': 'near-surface air temperature   [degK]', 'storage': '99'}

Unique output intervals [hours]: [3.]

TEMP0 range [K]: 208.4306182861328 to 315.99871826171875
TEMP0  global mean [K]: 279.9093322753906

Unique output intervals [hours]: [3.]
First time: 1989-01-01T00:00:00.000000000
Last time: 1991-12-31T21:00:00.000000000


In [6]:
# Build tas

tas = ds[SPEEDY_VARIABLE].rename(ACCESS_VARIABLE).astype("float32")

if SHIFT_LONGITUDE_TO_MINUS180_180:
    tas = tas.assign_coords(lon=((tas.lon + 180.0) % 360.0) - 180.0).sortby("lon")

if SORT_LATITUDE_NORTH_TO_SOUTH:
    tas = tas.sortby("lat", ascending=False)

tas.attrs = {
    "standard_name": "air_temperature",
    "long_name": "Near-Surface Air Temperature",
    "comment": "Near-surface air temperature from SPEEDY TEMP0",
    "units": "K",
    "cell_methods": "area: mean time: point",
    "source_variable": "TEMP0",
    "source_model": "SPEEDY",
    "mapping_note": "SPEEDY TEMP0 -> ACCESS-OM2 tas",
}

tas_ds = tas.to_dataset()

# JRA55-do 3hrPt convention: point time with ±1.5 h bounds
time = tas_ds.time
tas_ds["time_bnds"] = xr.DataArray(
    np.stack([(time - np.timedelta64(90, "m")).values,
              (time + np.timedelta64(90, "m")).values], axis=1),
    dims=("time", "bnds"), coords={"time": time, "bnds": [0, 1]}
)

tas_ds["lat"].attrs.update({"standard_name": "latitude", "long_name": "Latitude", "units": "degrees_north", "axis": "Y"})
tas_ds["lon"].attrs.update({"standard_name": "longitude", "long_name": "Longitude", "units": "degrees_east", "axis": "X"})
tas_ds["time"].attrs.update({"standard_name": "time", "long_name": "time", "axis": "T", "bounds": "time_bnds"})

tas_ds.attrs = {
    "Conventions": "CF-1.7",
    "title": "SPEEDY forcing for ACCESS-OM2",
    "source": "SPEEDY model output",
    "frequency": "3hrPt",
    "history": "Created from SPEEDY TEMP0 and exported as tas",
    "comment": "3-hourly point near-surface air temperature forcing following JRA55-do temporal convention.",
}

tas_ds

<xarray.Dataset> Size: 162MB
Dimensions:    (time: 8760, lat: 48, lon: 96, bnds: 2)
Coordinates:
  * time       (time) datetime64[ns] 70kB 1989-01-01 ... 1991-12-31T21:00:00
  * lat        (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon        (lon) float32 384B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
  * bnds       (bnds) int64 16B 0 1
Data variables:
    tas        (time, lat, lon) float32 161MB dask.array<chunksize=(1, 48, 96), meta=np.ndarray>
    time_bnds  (time, bnds) datetime64[ns] 140kB 1988-12-31T22:30:00 ... 1991...
Attributes:
    Conventions:  CF-1.7
    title:        SPEEDY forcing for ACCESS-OM2
    source:       SPEEDY model output
    frequency:    3hrPt
    history:      Created from SPEEDY TEMP0 and exported as tas
    comment:      3-hourly point near-surface air temperature forcing followi...

In [7]:
# Basic validation

required_dims = ("time", "lat", "lon")

if tas.dims != required_dims:
    raise ValueError(f"Expected tas dimensions {required_dims}, got {tas.dims}")

if tas.attrs.get("units") != "K":
    raise ValueError("tas must be stored in kelvin")

if tas.attrs.get("cell_methods") != "area: mean time: point":
    raise ValueError(f"Unexpected cell_methods: {tas.attrs.get('cell_methods')}")

if tas_ds.attrs.get("frequency") != "3hrPt":
    raise ValueError(f"Unexpected frequency: {tas_ds.attrs.get('frequency')}")

if not np.issubdtype(tas.dtype, np.floating):
    raise TypeError(f"tas must be floating point, got {tas.dtype}")

# Missing/invalid values
if not bool(np.isfinite(tas).all().compute()):
    raise ValueError("tas contains NaN or infinite values")

undef = ds.attrs.get("undef", 9.999e19)
invalid_count = int((np.abs(tas) >= abs(undef) * 0.9).sum().compute())
if invalid_count:
    raise ValueError(f"Found {invalid_count} values close to SPEEDY undef={undef}")

# Time axis: JRA55-do 3hrPt convention
dt_hours = np.diff(tas.time.values) / np.timedelta64(1, "h")
if not np.all(dt_hours == 3):
    raise ValueError(f"Expected 3-hourly time axis, got {np.unique(dt_hours)} h")

if "time_bnds" not in tas_ds:
    raise ValueError("time_bnds is missing")

bounds = tas_ds["time_bnds"]
width_hours = (bounds[:, 1] - bounds[:, 0]).values / np.timedelta64(1, "h")
midpoints = bounds[:, 0].values + (bounds[:, 1].values - bounds[:, 0].values) / 2

if not np.all(width_hours == 3):
    raise ValueError("time_bnds must be exactly 3 hours wide")

if not np.array_equal(midpoints, tas.time.values):
    raise ValueError("tas time must be the midpoint of time_bnds")

# Broad physical sanity check
tas_min = float(tas.min().compute())
tas_max = float(tas.max().compute())

if not (150.0 < tas_min < 350.0):
    print(f"WARNING: unusual minimum tas={tas_min:.3f} K")

if not (200.0 < tas_max < 400.0):
    print(f"WARNING: unusual maximum tas={tas_max:.3f} K")

print("Validation passed")
print(f"tas range: {tas_min:.3f} to {tas_max:.3f} K")
print("time:", tas.time.values[0], "to", tas.time.values[-1])
print("intervals [h]:", np.unique(dt_hours))
print("grid:", tas.sizes["lat"], "x", tas.sizes["lon"])

Validation passed
tas range: 208.431 to 315.999 K
time: 1989-01-01T00:00:00.000000000 to 1991-12-31T21:00:00.000000000
intervals [h]: [3.]
grid: 48 x 96


In [8]:
# 5. NetCDF encoding

field_encoding = {
    "dtype": "float32", "zlib": True, "complevel": 4, "shuffle": True,
    "_FillValue": np.float32(1.0e20),
    "chunksizes": (1, tas.sizes["lat"], tas.sizes["lon"]),
}

encoding = {
    ACCESS_VARIABLE: field_encoding,
    "time": {"dtype": "float64", "units": "days since 1900-01-01 00:00:00", "calendar": "gregorian", "_FillValue": None},
    "time_bnds": {"dtype": "float64", "units": "days since 1900-01-01 00:00:00", "calendar": "gregorian", "_FillValue": None},
    "lat": {"dtype": "float64", "_FillValue": None},
    "lon": {"dtype": "float64", "_FillValue": None},
}

encoding

{'tas': {'dtype': 'float32',
  'zlib': True,
  'complevel': 4,
  'shuffle': True,
  '_FillValue': np.float32(1e+20),
  'chunksizes': (1, 48, 96)},
 'time': {'dtype': 'float64',
  'units': 'days since 1900-01-01 00:00:00',
  'calendar': 'gregorian',
  '_FillValue': None},
 'time_bnds': {'dtype': 'float64',
  'units': 'days since 1900-01-01 00:00:00',
  'calendar': 'gregorian',
  '_FillValue': None},
 'lat': {'dtype': 'float64', '_FillValue': None},
 'lon': {'dtype': 'float64', '_FillValue': None}}

In [9]:
# Write NetCDF files

written_files = []

if WRITE_ONE_FILE_PER_YEAR:
    years = np.unique(tas_ds.time.dt.year.values)

    for year in years:
        yearly = tas_ds.sel(time=str(int(year)))

        if yearly.sizes["time"] != 2920:
            raise ValueError(f"{year}: expected 2920 3-hourly records, got {yearly.sizes['time']}")

        output_file = TAS_OUT_DIR / f"tas_SPEEDY_{int(year)}.nc"

        yearly.to_netcdf(
            output_file,
            mode="w",
            format="NETCDF4",
            engine="netcdf4",
            unlimited_dims=["time"],
            encoding=encoding,
        )

        written_files.append(output_file)
        print(f"Wrote {output_file.name}: {yearly.sizes['time']} records, {output_file.stat().st_size / 1024**2:.2f} MB")

else:
    output_file = TAS_OUT_DIR / "tas_SPEEDY_all_years.nc"

    tas_ds.to_netcdf(
        output_file,
        mode="w",
        format="NETCDF4",
        engine="netcdf4",
        unlimited_dims=["time"],
        encoding=encoding,
    )

    written_files.append(output_file)
    print(f"Wrote {output_file.name}: {tas_ds.sizes['time']} records, {output_file.stat().st_size / 1024**2:.2f} MB")

written_files

Wrote tas_SPEEDY_1989.nc: 2920 records, 32.11 MB
Wrote tas_SPEEDY_1990.nc: 2920 records, 32.17 MB
Wrote tas_SPEEDY_1991.nc: 2920 records, 32.17 MB


[PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/tas_SPEEDY_1989.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/tas_SPEEDY_1990.nc'),
 PosixPath('/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/access_forcing/tas_SPEEDY_1991.nc')]

In [10]:
# Reopen and verify output

if not written_files:
    raise RuntimeError("No NetCDF files were written")

check_file = written_files[0]

with xr.open_dataset(check_file, decode_times=True) as check:
    print(check)
    print("\nVariable attributes:", check[ACCESS_VARIABLE].attrs)
    print("\nEncoding:", check[ACCESS_VARIABLE].encoding)
    print("\nTime:", check.time.values[0], "to", check.time.values[-1])

    if check.sizes["time"] != 2920:
        raise ValueError(f"Expected 2920 records, got {check.sizes['time']}")

    if check.attrs.get("frequency") != "3hrPt":
        raise ValueError(f"Unexpected frequency: {check.attrs.get('frequency')}")

    if "time_bnds" not in check:
        raise ValueError("time_bnds is missing")

    dt_hours = np.diff(check.time.values) / np.timedelta64(1, "h")
    width_hours = (check.time_bnds[:, 1] - check.time_bnds[:, 0]).values / np.timedelta64(1, "h")
    midpoints = check.time_bnds[:, 0].values + (check.time_bnds[:, 1].values - check.time_bnds[:, 0].values) / 2

    if not np.all(dt_hours == 3):
        raise ValueError(f"Unexpected time intervals: {np.unique(dt_hours)} h")
    if not np.all(width_hours == 3):
        raise ValueError(f"Unexpected time_bnds widths: {np.unique(width_hours)} h")
    if not np.array_equal(midpoints, check.time.values):
        raise ValueError("time is not the midpoint of time_bnds")

    if check.time.encoding.get("units") != "days since 1900-01-01":
        raise ValueError(f"Unexpected time units: {check.time.encoding.get('units')}")
    if check.time.encoding.get("calendar") != "gregorian":
        raise ValueError(f"Unexpected calendar: {check.time.encoding.get('calendar')}")

    tas_min = float(check[ACCESS_VARIABLE].min())
    tas_max = float(check[ACCESS_VARIABLE].max())
    print(f"\nRange [K]: {tas_min:.3f} to {tas_max:.3f}")

    source_first = tas.sel(time=check.time.values[0]).compute()
    output_first = check[ACCESS_VARIABLE].isel(time=0).load()
    max_abs_difference = float(np.abs(source_first - output_first).max())

    print("Maximum absolute difference after NetCDF round trip:", max_abs_difference)

    if max_abs_difference != 0.0:
        raise ValueError("NetCDF round trip changed tas values")

print("Output verification passed")

<xarray.Dataset> Size: 54MB
Dimensions:    (time: 2920, lat: 48, lon: 96, bnds: 2)
Coordinates:
  * time       (time) datetime64[ns] 23kB 1989-01-01 ... 1989-12-31T21:00:00
  * lat        (lat) float64 384B -87.16 -83.48 -79.78 ... 79.78 83.48 87.16
  * lon        (lon) float64 768B 0.0 3.75 7.5 11.25 ... 345.0 348.8 352.5 356.2
  * bnds       (bnds) int64 16B 0 1
Data variables:
    tas        (time, lat, lon) float32 54MB ...
    time_bnds  (time, bnds) datetime64[ns] 47kB ...
Attributes:
    Conventions:  CF-1.7
    title:        SPEEDY forcing for ACCESS-OM2
    source:       SPEEDY model output
    frequency:    3hrPt
    history:      Created from SPEEDY TEMP0 and exported as tas
    comment:      3-hourly point near-surface air temperature forcing followi...

Variable attributes: {'standard_name': 'air_temperature', 'long_name': 'Near-Surface Air Temperature', 'comment': 'Near-surface air temperature from SPEEDY TEMP0', 'units': 'K', 'cell_methods': 'area: mean time: point', 'so